In [1]:
%matplotlib inline

In [2]:
import xarray as xr
import matplotlib.pyplot as plt

In [5]:
models = [('ACCESS-ESM1-5','BARPA-C'),
          ('EC-Earth3','BARPA-C'),
          ('EC-Earth3','BARPA-R'),
          ('obs','radar_rainrate_level2')]

radars = [2,4,9,22,49,54,63,66,71]

In [6]:
radar_names = {}
for radar in radars:
    tmp = xr.open_mfdataset(f"/g/data/rq0/level_2/{radar}/REFLECTIVITY/{radar}_20110822_reflectivity.nc")
    radar_names[radar] = tmp.attrs['instrument_name']

In [7]:
outdir = "/g/data/tp28/dev/eh6215/model_evaluation_suite/output"
path = "{outdir}/{gcm}/historical/r*i1p1f1/{rcm}/radar_distributions/Radar{n}_*.nc"

In [8]:
data = {}
for radar in radars:
    data[radar_names[radar]] = {}
    for (gcm,rcm) in models:
        name = f"{gcm}_{rcm}"
        data[radar_names[radar]][name] = xr.open_mfdataset(path.format(outdir=outdir,gcm=gcm,rcm=rcm,n=radar))

In [10]:
plot_kwargs = {'ACCESS-ESM1-5_BARPA-C':{'color':'tab:blue','ls':':'},
               'ACCESS-ESM1-5_BARPA-R':{'color':'tab:red','ls':':'},
               'EC-Earth3_BARPA-C':{'color':'tab:blue','ls':'--'},
               'EC-Earth3_BARPA-R':{'color':'tab:red','ls':'--'},
               'obs_radar_rainrate_level2':{'color':"k",'ls':'-'}}

In [12]:
def radarplot(season,radar):
    plt.figure(figsize=(4.5,4.5))
    for name in data[radar]:
        var = [x for x in data[radar][name].variables][-1]
        tmp = data[radar][name][var]
        if season != 'annual':
            tmp = tmp.sel(time=tmp.time.dt.season==season)
        tmp.sum('time').plot(label=name,**plot_kwargs[name])
    plt.title(f"Rain rate distribution at {radar} radar")
    plt.legend()
    plt.semilogy()

In [13]:
from ipywidgets import interactive
import matplotlib.pyplot as plt
import numpy as np


interactive_plot = interactive(radarplot, season = ['annual','DJF','MAM','JJA','SON'],radar=list(radar_names.values()))
output = interactive_plot.children[-1]
interactive_plot

interactive(children=(Dropdown(description='season', options=('annual', 'DJF', 'MAM', 'JJA', 'SON'), value='an…

In [38]:
data[22]['obs_radar_rainrate_level2']

<xarray.Dataset>
Dimensions:             (time: 108, rainrate_bin: 209)
Coordinates:
  * time                (time) datetime64[ns] 2013-01-31 ... 2021-12-31
  * rainrate_bin        (rainrate_bin) float64 1.0 3.0 5.0 ... 413.0 415.0 417.0
Data variables:
    histogram_rainrate  (time, rainrate_bin) int64 dask.array<chunksize=(108, 209), meta=np.ndarray>

In [ ]:
p